In [2]:
# ============================================================
# FINAL ML PROJECT - CLASSIFICATION
# Alissa Canesim Bento
# HR Analytics: Job Change of Data Scientists
# ============================================================

# 1. IMPORT LIBRARIES

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

from sklearn.linear_model import LogisticRegression, BayesianRidge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

pd.set_option("display.max_columns", None)


In [3]:
# 2. LOAD DATASET

from google.colab import files
uploaded = files.upload()

df = pd.read_csv("aug_train.csv")

print("Dataset shape:", df.shape)
display(df.head())

Saving aug_train.csv to aug_train.csv
Dataset shape: (19158, 14)


,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,8949,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevent experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


In [4]:
# 3. DEFINE TARGET AND SPLIT DATA 70/15/15

TARGET_COL = "target"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# First split: 70% train, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape, round(len(X_train) / len(df), 2))
print("Validation:", X_val.shape, round(len(X_val) / len(df), 2))
print("Test:", X_test.shape, round(len(X_test) / len(df), 2))

Train: (13410, 13) 0.7
Validation: (2874, 13) 0.15
Test: (2874, 13) 0.15


In [5]:
# 4. MIDTERM PREPROCESSING LOGIC

def parse_experience(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "<1":
        return 0
    if s == ">20":
        return 21
    return pd.to_numeric(s, errors="coerce")


def parse_last_new_job(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s == "never":
        return 0
    if s == ">4":
        return 5
    return pd.to_numeric(s, errors="coerce")


def clean_company_size(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "10/49":
        return "10-49"
    return s


education_map = {
    "Primary School": 0,
    "High School": 1,
    "Graduate": 2,
    "Masters": 3,
    "Phd": 4
}

company_size_map = {
    "<10": 0,
    "10-49": 1,
    "50-99": 2,
    "100-500": 3,
    "500-999": 4,
    "1000-4999": 5,
    "5000-9999": 6,
    "10000+": 7
}

relevant_experience_map = {
    "No relevent experience": 0,
    "Has relevent experience": 1
}


def map_experience_level(x):
    if pd.isna(x):
        return "Missing"
    if x <= 2:
        return "Junior"
    elif x <= 5:
        return "Mid"
    elif x <= 10:
        return "Senior"
    else:
        return "Expert"


def add_features(frame):
    d = frame.copy()

    d["company_size"] = d["company_size"].apply(clean_company_size)

    d["relevent_experience"] = d["relevent_experience"].map(relevant_experience_map)
    d["education_level"] = d["education_level"].map(education_map)
    d["company_size"] = d["company_size"].map(company_size_map)

    d["experience_num"] = d["experience"].apply(parse_experience)
    d["last_new_job_num"] = d["last_new_job"].apply(parse_last_new_job)

    d["training_per_experience"] = d["training_hours"] / (d["experience_num"] + 1)
    d["training_hours_log1p"] = np.log1p(d["training_hours"].clip(lower=0))

    d["experience_level"] = d["experience_num"].apply(map_experience_level)

    d = d.drop(columns=["enrollee_id", "experience", "last_new_job"])

    return d


X_train_ready = add_features(X_train)
X_val_ready = add_features(X_val)
X_test_ready = add_features(X_test)

print("Train ready:", X_train_ready.shape)
print("Validation ready:", X_val_ready.shape)
print("Test ready:", X_test_ready.shape)

Train ready: (13410, 15)
Validation ready: (2874, 15)
Test ready: (2874, 15)


In [6]:
# 5. BUILD PREPROCESSING PIPELINE

numeric_features = X_train_ready.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train_ready.select_dtypes(include=["object", "category"]).columns.tolist()

city_categories = [f"city_{i}" for i in range(1, 181)]

ordinal_mappings = {
    "gender": ["Male", "Female", "Other"],
    "enrolled_university": ["no_enrollment", "Part time course", "Full time course"],
    "experience_level": ["Missing", "Junior", "Mid", "Senior", "Expert"],
    "city": city_categories
}

ordinal_features = [col for col in categorical_features if col in ordinal_mappings]
nominal_features = [col for col in categorical_features if col not in ordinal_mappings]

ordinal_categories = [ordinal_mappings[col] for col in ordinal_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=ordinal_categories, handle_unknown="use_encoded_value", unknown_value=-1))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("ord", ordinal_transformer, ordinal_features),
        ("cat", categorical_transformer, nominal_features)
    ]
)

X_train_clean = preprocessor.fit_transform(X_train_ready)
X_val_clean = preprocessor.transform(X_val_ready)
X_test_clean = preprocessor.transform(X_test_ready)

print("Processed train:", X_train_clean.shape)
print("Processed validation:", X_val_clean.shape)
print("Processed test:", X_test_clean.shape)

Processed train: (13410, 25)
Processed validation: (2874, 25)
Processed test: (2874, 25)


In [7]:
# 6. TRAIN REQUIRED CLASSIFICATION MODELS

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
        min_samples_leaf=2
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "Support Vector Classifier": SVC(probability=True, random_state=42, class_weight="balanced")
}

In [8]:
# 7. EVALUATION FUNCTION

def evaluate_model(model, X_data, y_true, dataset_name):
    y_pred = model.predict(X_data)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_data)[:, 1]
    else:
        y_proba = y_pred

    return {
        "Dataset": dataset_name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4),
        "F1 Score": round(f1_score(y_true, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_true, y_proba), 4)
    }

In [9]:
# 8. TRAIN MODELS AND COMPARE VALIDATION + TEST RESULTS

all_results = []

for name, model in models.items():
    model.fit(X_train_clean, y_train)

    val_metrics = evaluate_model(model, X_val_clean, y_val, "Validation")
    val_metrics["Model"] = name
    all_results.append(val_metrics)

    test_metrics = evaluate_model(model, X_test_clean, y_test, "Test")
    test_metrics["Model"] = name
    all_results.append(test_metrics)

results_df = pd.DataFrame(all_results)
results_df = results_df[["Model", "Dataset", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]]

display(results_df)

,Model,Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,Validation,0.7122,0.4462,0.6360,0.5244,0.7321
1,Logistic Regression,Test,0.7074,0.4405,0.6466,0.5241,0.7334
2,Decision Tree,Validation,0.7161,0.4298,0.4226,0.4262,0.6181
3,Decision Tree,Test,0.7227,0.4444,0.4525,0.4484,0.6320
4,Random Forest,Validation,0.7878,0.5790,0.5467,0.5624,0.7950
5,Random Forest,Test,0.7735,0.5455,0.5447,0.5451,0.7837
6,Gradient Boosting,Validation,0.7888,0.6054,0.4407,0.5101,0.8061
7,Gradient Boosting,Test,0.7756,0.5661,0.4246,0.4852,0.7891
8,K-Nearest Neighbors,Validation,0.7512,0.5018,0.3849,0.4357,0.7210
9,K-Nearest Neighbors,Test,0.7491,0.4958,0.4106,0.4492,0.7132


In [10]:
# 9. CHOOSE TOP 3 MODELS BASED ON VALIDATION ROC-AUC

validation_results = results_df[results_df["Dataset"] == "Validation"].sort_values(
    by="ROC-AUC",
    ascending=False
)

display(validation_results)

top_3_model_names = validation_results.head(3)["Model"].tolist()
print("Top 3 models:", top_3_model_names)

,Model,Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
6,Gradient Boosting,Validation,0.7888,0.6054,0.4407,0.5101,0.8061
4,Random Forest,Validation,0.7878,0.5790,0.5467,0.5624,0.7950
0,Logistic Regression,Validation,0.7122,0.4462,0.6360,0.5244,0.7321
8,K-Nearest Neighbors,Validation,0.7512,0.5018,0.3849,0.4357,0.7210
10,Support Vector Classifier,Validation,0.7370,0.4734,0.4840,0.4786,0.6923
2,Decision Tree,Validation,0.7161,0.4298,0.4226,0.4262,0.6181


Top 3 models: ['Gradient Boosting', 'Random Forest', 'Logistic Regression']


In [11]:
# 10. VOTING CLASSIFIER ENSEMBLE

top_3_estimators = []

for name in top_3_model_names:
    top_3_estimators.append((name, models[name]))

voting_model = VotingClassifier(
    estimators=top_3_estimators,
    voting="soft"
)

voting_model.fit(X_train_clean, y_train)

voting_results = []

val_metrics = evaluate_model(voting_model, X_val_clean, y_val, "Validation")
val_metrics["Model"] = "Voting Ensemble"
voting_results.append(val_metrics)

test_metrics = evaluate_model(voting_model, X_test_clean, y_test, "Test")
test_metrics["Model"] = "Voting Ensemble"
voting_results.append(test_metrics)

voting_results_df = pd.DataFrame(voting_results)
voting_results_df = voting_results_df[["Model", "Dataset", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]]

display(voting_results_df)

,Model,Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Voting Ensemble,Validation,0.7898,0.5879,0.5272,0.5559,0.7992
1,Voting Ensemble,Test,0.7791,0.5595,0.5321,0.5455,0.7873


In [12]:
# 11. BAYESIAN ENSEMBLE MODEL
# This model uses the prediction probabilities from the top 3 models
# and combines them using Bayesian Ridge.

# Generate probability predictions from the top 3 models
train_meta_features = []
val_meta_features = []
test_meta_features = []

for name in top_3_model_names:
    model = models[name]

    train_meta_features.append(model.predict_proba(X_train_clean)[:, 1])
    val_meta_features.append(model.predict_proba(X_val_clean)[:, 1])
    test_meta_features.append(model.predict_proba(X_test_clean)[:, 1])

X_train_meta = np.column_stack(train_meta_features)
X_val_meta = np.column_stack(val_meta_features)
X_test_meta = np.column_stack(test_meta_features)

bayesian_ensemble = BayesianRidge()
bayesian_ensemble.fit(X_train_meta, y_train)

val_bayes_proba = bayesian_ensemble.predict(X_val_meta)
test_bayes_proba = bayesian_ensemble.predict(X_test_meta)

# Keep probabilities between 0 and 1
val_bayes_proba = np.clip(val_bayes_proba, 0, 1)
test_bayes_proba = np.clip(test_bayes_proba, 0, 1)

val_bayes_pred = (val_bayes_proba >= 0.5).astype(int)
test_bayes_pred = (test_bayes_proba >= 0.5).astype(int)

bayesian_results = [
    {
        "Model": "Bayesian Ensemble",
        "Dataset": "Validation",
        "Accuracy": round(accuracy_score(y_val, val_bayes_pred), 4),
        "Precision": round(precision_score(y_val, val_bayes_pred), 4),
        "Recall": round(recall_score(y_val, val_bayes_pred), 4),
        "F1 Score": round(f1_score(y_val, val_bayes_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_val, val_bayes_proba), 4)
    },
    {
        "Model": "Bayesian Ensemble",
        "Dataset": "Test",
        "Accuracy": round(accuracy_score(y_test, test_bayes_pred), 4),
        "Precision": round(precision_score(y_test, test_bayes_pred), 4),
        "Recall": round(recall_score(y_test, test_bayes_pred), 4),
        "F1 Score": round(f1_score(y_test, test_bayes_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, test_bayes_proba), 4)
    }
]

bayesian_results_df = pd.DataFrame(bayesian_results)
display(bayesian_results_df)

,Model,Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Bayesian Ensemble,Validation,0.7672,0.5424,0.4282,0.4786,0.7455
1,Bayesian Ensemble,Test,0.7582,0.5175,0.4330,0.4715,0.7412


In [13]:
# 12. FINAL COMPARISON TABLE

final_results_df = pd.concat(
    [results_df, voting_results_df, bayesian_results_df],
    ignore_index=True
)

final_results_df = final_results_df.sort_values(
    by=["Dataset", "ROC-AUC", "F1 Score"],
    ascending=[True, False, False]
)

display(final_results_df)

,Model,Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
7,Gradient Boosting,Test,0.7756,0.5661,0.4246,0.4852,0.7891
13,Voting Ensemble,Test,0.7791,0.5595,0.5321,0.5455,0.7873
5,Random Forest,Test,0.7735,0.5455,0.5447,0.5451,0.7837
15,Bayesian Ensemble,Test,0.7582,0.5175,0.4330,0.4715,0.7412
1,Logistic Regression,Test,0.7074,0.4405,0.6466,0.5241,0.7334
9,K-Nearest Neighbors,Test,0.7491,0.4958,0.4106,0.4492,0.7132
11,Support Vector Classifier,Test,0.7220,0.4446,0.4651,0.4546,0.6852
3,Decision Tree,Test,0.7227,0.4444,0.4525,0.4484,0.6320
6,Gradient Boosting,Validation,0.7888,0.6054,0.4407,0.5101,0.8061
12,Voting Ensemble,Validation,0.7898,0.5879,0.5272,0.5559,0.7992


In [14]:
# 13. SAVE RESULTS TABLE AS CSV

final_results_df.to_csv("final_model_comparison_results.csv", index=False)

print("Saved final comparison table as final_model_comparison_results.csv")

Saved final comparison table as final_model_comparison_results.csv


In [15]:
# 14. SIMPLE FINAL INTERPRETATION

best_validation_model = validation_results.iloc[0]["Model"]

print("Final Project Summary")
print("----------------------")
print("This project uses the HR Analytics Job Change dataset.")
print("The target variable is 'target', which makes this a binary classification problem.")
print("The data was split into 70% training, 15% validation, and 15% testing.")
print("The preprocessing logic was reused from the midterm project.")
print("The required classification models were trained and compared using accuracy, precision, recall, F1 score, and ROC-AUC.")
print("Based on validation ROC-AUC, the best individual model was:", best_validation_model)
print("The Voting Ensemble and Bayesian Ensemble were also evaluated and compared against the individual models.")

Final Project Summary
----------------------
This project uses the HR Analytics Job Change dataset.
The target variable is 'target', which makes this a binary classification problem.
The data was split into 70% training, 15% validation, and 15% testing.
The preprocessing logic was reused from the midterm project.
The required classification models were trained and compared using accuracy, precision, recall, F1 score, and ROC-AUC.
Based on validation ROC-AUC, the best individual model was: Gradient Boosting
The Voting Ensemble and Bayesian Ensemble were also evaluated and compared against the individual models.
